In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# 데이터셋 로딩, RoBERTa 학습, 성능 평가에 필요한 라이브러리를 설치합니다.
!pip -q install transformers datasets accelerate scikit-learn

In [1]:
# 데이터 처리, 시각화, PyTorch 학습에 사용할 모듈을 불러옵니다.
import re
import ast
import json
import html
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from pathlib import Path
from IPython.display import display, HTML
from datasets import load_dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

c:\VSGit\Personal_Mini_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# RoBERTa.ipynb과 같은 Google Drive 폴더에 학습 결과를 저장합니다.
MODEL_DIR = Path("/content/drive/MyDrive/HC3_ko")
BEST_MODEL_DIR = MODEL_DIR / "best_model"


In [ ]:

MODEL_NAME = "klue/roberta-base"      # Hugging Face 사전학습 모델
DATASET_NAME = "nayohan/HC3-ko"      # Hugging Face 데이터셋

MAX_LEN = 256                        # 입력 최대 토큰 수; 긴 글은 잘림 (현재 모델 한계 512)
MAX_PAIRS = 4000                     # 사용할 질문-답변 쌍 최대 수

# MIN_CHARS = 20                     # 짧은 답변을 제외하려면 아래 필터와 함께 주석 해제
# human_answers = [a for a in human_answers if len(a) >= MIN_CHARS]
# chatgpt_answers = [a for a in chatgpt_answers if len(a) >= MIN_CHARS]

BATCH_SIZE = 16                      # 한 번에 학습할 글 수
NUM_WORKERS = 0                     # DataLoader 병렬 작업 수

EPOCHS = 3                           # 전체 데이터 반복 횟수
LR = 2e-5                            # 학습률
WEIGHT_DECAY = 0.01                 # 가중치 감쇠
WARMUP_RATIO = 0.10                 # 전체 학습 단계 중 Warmup 비율

SUSPICION_THRESHOLD = 0.50          # 문장별 의심 구역 표시 기준
CONTEXT_RADIUS = 1                  # 각 문장 점수 계산 시 포함할 앞뒤 문장 수

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    DEVICE = torch.device("xpu")
else:
    DEVICE = torch.device("cpu")

print("DEVICE:", DEVICE)

In [ ]:
# 설정 셀의 모델 이름으로 Tokenizer를 불러와 글을 모델 입력으로 변환할 준비
# 모델과 함께 공개된 Tokenizer를 사용해야 토큰 ID가 맞음
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("모델:", MODEL_NAME)
print("Tokenizer:", type(tokenizer).__name__)
print("Vocab 크기:", tokenizer.vocab_size)

# 데이터셋 준비

In [ ]:
# Hugging Face 원본 데이터셋의 구조와 질문 수를 먼저 확인합니다.
raw_dataset = load_dataset(
    DATASET_NAME,
    split="train",  # train 분할 하나만 제공됨
)

print(raw_dataset)
print("전체 질문 행 수:", len(raw_dataset))

# 이후 학습에는 질문과 두 종류의 답변만 사용합니다.
raw_dataset = raw_dataset.select_columns(
    ["question", "human_answers", "chatgpt_answers"]
)

### 학습에 사용하는 HC3-ko 컬럼

In [ ]:
# 문자열로 저장된 답변 목록을 실제 리스트로 바꾸어 답변만 추출합니다.
def parse_answers(value):
    # 데이터가 이미 목록으로 읽힌 경우
    if isinstance(value, list):
        answers = value

    elif isinstance(value, str):
        value = value.strip()

        if not value:
            return []

        if value.startswith("["):
            # "['답변1', '답변2']"처럼 저장된 문자열을 목록으로 변환
            try:
                answers = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return []
        else:
            # 일반 문자열이면 답변 하나로 처리
            answers = [value]

    else:
        return []

    if not isinstance(answers, list):
        return []

    # 빈 답변과 문자열이 아닌 값은 제외
    return [
        answer.strip()
        for answer in answers
        if isinstance(answer, str) and answer.strip()
    ]

### 지름길(Shortcut) 제거


In [ ]:
# 두 클래스 답변에 같은 규칙을 적용해 형식 흔적(URL 자리표시자, 구두점 공백)을 정리합니다.
def clean_text(text):
    text = html.unescape(text)

    # [ 링크 글자 ] ( URL_0 ) -> 링크 글자
    text = re.sub(r"\[\s*([^\]]*?)\s*\]\s*\(\s*URL_\d+\s*\)", r"\1", text)

    # 남은 URL 자리표시자 제거
    text = re.sub(r"\(?\s*URL_\d+\s*\)?", " ", text)

    # 구두점/닫는 괄호 앞 공백, 여는 괄호 뒤 공백 제거
    text = re.sub(r"\s+([.,!?;:%)\]])", r"\1", text)
    text = re.sub(r"([(\[])\s+", r"\1", text)

    # 연속된 공백을 하나로
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)

    return text.strip()



In [ ]:
# 각 질문에서 사람 답변과 ChatGPT 답변을 한 개씩 골라 한 쌍으로 만듦

pairs = []

for row in raw_dataset:
    human_answers = [
        clean_text(answer)
        for answer in parse_answers(row["human_answers"])
    ]
    chatgpt_answers = [
        clean_text(answer)
        for answer in parse_answers(row["chatgpt_answers"])
    ]

    # human_answers = [a for a in human_answers if len(a) >= MIN_CHARS]
    # chatgpt_answers = [a for a in chatgpt_answers if len(a) >= MIN_CHARS]

    # 정리 후 비어 버린 답변만 제외합니다.
    human_answers = [a for a in human_answers if a]
    chatgpt_answers = [a for a in chatgpt_answers if a]

    # 사람과 ChatGPT 답변이 모두 있는 질문만 사용
    if not human_answers or not chatgpt_answers:
        continue

    # 클래스 균형을 위해 두 클래스에서 첫 번째 답변 하나씩 선택
    pairs.append({
        "question": str(row["question"]).strip(),
        "human_text": human_answers[0],
        "chatgpt_text": chatgpt_answers[0],
    })

pairs_df = pd.DataFrame(pairs)

print("사용 가능한 질문-답변 쌍:", len(pairs_df))

In [ ]:
# 설정 셀의 MAX_PAIRS만큼 질문 쌍을 사용해 학습 시간을 조절합니다.
if len(pairs_df) > MAX_PAIRS:
    pairs_df = pairs_df.sample(n=MAX_PAIRS).reset_index(drop=True)

print("사용 질문 쌍:", len(pairs_df))
print("전체 답변 수:", len(pairs_df) * 2)

In [ ]:
# 셀 생략 가능
# 글자 수와 토큰 수를 비교해 MAX_LEN가 적당한지 확인합니다.
pairs_df["human_length"] = pairs_df["human_text"].str.len()
pairs_df["chatgpt_length"] = pairs_df["chatgpt_text"].str.len()

display(
    pairs_df[["human_length", "chatgpt_length"]].describe()
)

for column, name in [("human_text", "사람"), ("chatgpt_text", "ChatGPT")]:
    token_lengths = np.array([
        len(ids)
        for ids in tokenizer(pairs_df[column].tolist())["input_ids"]
    ])

    print(
        f"{name:8s} 토큰 수 | "
        f"중앙값 {np.median(token_lengths):.0f} | "
        f"90% {np.percentile(token_lengths, 90):.0f} | "
        f"MAX_LEN({MAX_LEN}) 초과 {np.mean(token_lengths > MAX_LEN) * 100:.1f}%"
    )

### 데이터 분할 (Train 80% / Validation 10% / Test 10%)

In [ ]:
# 같은 질문이 학습과 평가에 섞이지 않도록 질문 단위로 학습 데이터를 나눕니다.
splitter_1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,  # 전체 질문의 20%를 검증·테스트용으로 분리
)

train_idx, temp_idx = next(
    splitter_1.split(
        pairs_df,
        groups=pairs_df["question"],
    )
)

train_pairs = pairs_df.iloc[train_idx].reset_index(drop=True)
temp_pairs = pairs_df.iloc[temp_idx].reset_index(drop=True)

# 남은 질문을 다시 반씩 나누어 검증용과 최종 평가용으로 사용
splitter_2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,  # 분리된 20%를 반씩 나누어 검증·테스트 각 10%
)

val_idx, test_idx = next(
    splitter_2.split(
        temp_pairs,
        groups=temp_pairs["question"],
    )
)

val_pairs = temp_pairs.iloc[val_idx].reset_index(drop=True)
test_pairs = temp_pairs.iloc[test_idx].reset_index(drop=True)

print("Train 질문 쌍:", len(train_pairs))
print("Val   질문 쌍:", len(val_pairs))
print("Test  질문 쌍:", len(test_pairs))

In [ ]:
# 세 분할에 같은 질문이 중복되지 않는지 검사합니다.
train_questions = set(train_pairs["question"])
val_questions = set(val_pairs["question"])
test_questions = set(test_pairs["question"])

# 중복 질문이 있으면 평가가 과대 추정되므로 실행 중단
assert train_questions.isdisjoint(val_questions)
assert train_questions.isdisjoint(test_questions)
assert val_questions.isdisjoint(test_questions)

print("분할 사이에 겹치는 질문이 없습니다.")

In [ ]:
# 답변 쌍을 개별 글과 사람(0)/ChatGPT(1) 라벨로 바꿉니다.
LABEL_NAMES = ["HUMAN", "CHATGPT"]


def make_classification_df(pair_df):
    rows = []

    for _, row in pair_df.iterrows():
        rows.append({
            "text": row["human_text"],
            "label": 0,  # 사람 답변
        })
        rows.append({
            "text": row["chatgpt_text"],
            "label": 1,  # ChatGPT 답변
        })

    return pd.DataFrame(rows)


train_df = make_classification_df(train_pairs)
val_df = make_classification_df(val_pairs)
test_df = make_classification_df(test_pairs)

print("Train 글:", len(train_df))
print("Val   글:", len(val_df))
print("Test  글:", len(test_df))

# 3. Dataset / DataLoader

In [ ]:
# 글을 미리 토큰화해 두고, padding은 배치를 만들 때 수행하는 Dataset을 정의합니다.
class TextClassificationDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.labels = dataframe["label"].tolist()

        # padding 없이 잘라내기만 함 (padding은 collator가 배치 단위로 수행)
        self.encoded = tokenizer(
            dataframe["text"].tolist(),
            truncation=True,
            max_length=max_length,
            return_token_type_ids=False,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            "input_ids": self.encoded["input_ids"][index],
            "attention_mask": self.encoded["attention_mask"][index],
            "labels": self.labels[index],
        }

In [ ]:
# 학습/검증/평가 Dataset과 배치 단위 padding을 담당할 collator를 만듭니다.
train_dataset = TextClassificationDataset(train_df, tokenizer, MAX_LEN)
val_dataset = TextClassificationDataset(val_df, tokenizer, MAX_LEN)
test_dataset = TextClassificationDataset(test_df, tokenizer, MAX_LEN)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt",
)

print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))
print("Test :", len(test_dataset))

In [ ]:
# 설정 셀의 배치 크기와 작업 수로 DataLoader를 만듭니다.
loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    collate_fn=collator,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

train_loader = DataLoader(
    train_dataset,
    shuffle=True,   # 학습할 때만 순서를 섞음
    **loader_kwargs,
)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print("Train batches:", len(train_loader))
print("Val   batches:", len(val_loader))
print("Test  batches:", len(test_loader))

# 4. 모델 (RoBERTa + Classification Head)

In [ ]:
# 사전학습 RoBERTa에 2개 클래스용 분류 헤드를 붙여 모델을 준비합니다.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={i: name for i, name in enumerate(LABEL_NAMES)},
    label2id={name: i for i, name in enumerate(LABEL_NAMES)},
).to(DEVICE)

model

# **5. 학습 설정**
### Fine-tuning 하이퍼파라미터
- RoBERTa-base 미세조정 권장 범위: LR `1e-5 ~ 5e-5`, Epoch `2 ~ 4`, Warmup `6 ~ 10%`
- `bias`와 `LayerNorm` 가중치에는 weight decay를 적용하지 않음 (BERT 계열 표준 설정)
- ViT 예제는 Epoch마다 `CosineAnnealingLR.step()`을 했지만, 여기서는 **Warmup + Linear Decay를 배치마다** `step()`
    - 사전학습 가중치는 처음에 큰 LR을 받으면 망가지기 쉬워서 Warmup으로 LR을 천천히 올림
- 손실 함수는 모델 내부의 CrossEntropyLoss 사용 (`labels`를 넣으면 `outputs.loss` 반환)

In [ ]:
# 설정 셀의 학습 값으로 Optimizer와 Scheduler를 만듭니다.
# bias, LayerNorm은 weight decay에서 제외
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_parameters = [
    {
        "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LR)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print("Total steps :", total_steps)
print("Warmup steps:", warmup_steps)
print(optimizer)

### AMP(Automatic Mixed Precision)
- GPU에서는 mixed precision을 사용하며 연산 속도를 높이고 GPU 메모리 사용량을 줄일 수 있음
- 자동 혼합 정밀도
    - FP32(32비트 부동소수점): 정밀한 계산. 메모리를 더 많이 사용. 계산량이 큼
    - FP16(16비트 부동소수점): 낮은 정밀도. 메모리를 절약. 계산량이 적음
> 상황에 맞게 섞어 사용할 수 있는 기술이 Mixed Precision

In [ ]:
use_amp = (DEVICE.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print("AMP enabled: ", use_amp)

### 평가 지표
- Positive 클래스 = **ChatGPT(1)**
- **Precision**: ChatGPT라고 판정한 글 중 실제 ChatGPT 비율 → 낮으면 사람 글을 AI로 오판
- **Recall**: 실제 ChatGPT 글 중 잡아낸 비율 → 낮으면 AI 글을 놓침
- **F1**: Precision과 Recall의 조화 평균 → **Best model 선택 기준**

In [ ]:
# 정답과 예측을 비교해 전체 분류 지표를 계산합니다.
def compute_metrics(labels, predictions):
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
    }

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    all_labels = []
    all_predictions = []

    for batch in loader:
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}

        with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
            outputs = model(**batch)

        logits = outputs.logits.float()
        predictions = logits.argmax(dim=1)

        batch_size = batch["labels"].size(0)
        total_loss += outputs.loss.item() * batch_size
        total_samples += batch_size

        all_labels.extend(batch["labels"].cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())

    return {
        "loss": total_loss / total_samples,
        **compute_metrics(all_labels, all_predictions),
    }

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, device):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in loader:
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()  # Warmup/Linear scheduler는 배치마다 갱신

        batch_size = batch["labels"].size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples

# **6. 학습**

In [ ]:
# 설정 셀에서 선택한 폴더를 생성해 학습 결과를 저장할 준비를 합니다.
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("결과 저장 위치:", MODEL_DIR)

In [ ]:
history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": [],
}

best_f1 = -1.0
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        scaler,
        DEVICE,
    )

    metrics = evaluate(
        model,
        val_loader,
        DEVICE,
    )

    history["train_loss"].append(train_loss)
    for key in ["loss", "accuracy", "precision", "recall", "f1"]:
        history[f"val_{key}"].append(metrics[key])

    print(
        f"[{epoch:02d}/{EPOCHS}] "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={metrics['loss']:.4f} | "
        f"Acc={metrics['accuracy']*100:6.2f}% | "
        f"P={metrics['precision']:.4f} | "
        f"R={metrics['recall']:.4f} | "
        f"F1={metrics['f1']:.4f} | "
        f"lr={optimizer.param_groups[0]['lr']:.2e} | "
        f"{time.time()-start:.1f}s"
    )

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_epoch = epoch

        # Hugging Face 형식으로 모델, Tokenizer를 함께 저장
        model.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)

        print(f"  -> Best model 저장: {BEST_MODEL_DIR}")

print(f"\nBest Validation F1: {best_f1:.4f} (epoch {best_epoch})")

In [ ]:
# Epoch별 기록을 표로 정리합니다. (CSV 저장에서 사용하므로 실행 필요)
history_df = pd.DataFrame(history)
history_df.insert(0, "epoch", range(1, len(history_df) + 1))

display(history_df)

In [ ]:
epochs = history_df["epoch"]

plt.figure(figsize=(7, 4))
plt.plot(epochs, history_df["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs, history_df["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(epochs, history_df["val_f1"], marker="o", label="Validation F1")
plt.plot(epochs, history_df["val_precision"], marker="o", linestyle="--", label="Validation Precision")
plt.plot(epochs, history_df["val_recall"], marker="o", linestyle="--", label="Validation Recall")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Validation Performance (Positive = ChatGPT)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 7. Test 평가

In [ ]:
best_model = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL_DIR).to(DEVICE)
best_tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
best_model.eval()

print("Best epoch:", best_epoch)
print("Best validation F1:", f"{best_f1:.4f}")
print("클래스:", best_model.config.id2label)

In [ ]:
test_result = evaluate(best_model, test_loader, DEVICE)

print(f"Accuracy : {test_result['accuracy']:.4f}")
print(f"Precision: {test_result['precision']:.4f}")
print(f"Recall   : {test_result['recall']:.4f}")
print(f"F1-score : {test_result['f1']:.4f}")

In [ ]:
# 학습 기록과 Test 지표를 Drive에 저장해 나중에 다시 확인할 수 있게 합니다.
history_df.to_csv(
    MODEL_DIR / "training_history.csv",
    index=False,
    encoding="utf-8-sig",
)

results = {
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "max_pairs": MAX_PAIRS,
    "max_length": MAX_LEN,
    "epochs": EPOCHS,
    "lr": LR,
    "best_epoch": best_epoch,
    "best_validation_f1": best_f1,
    **{f"test_{key}": test_result[key] for key in ["accuracy", "precision", "recall", "f1"]},
}

with open(MODEL_DIR / "test_metrics.json", "w", encoding="utf-8") as file:
    json.dump(results, file, ensure_ascii=False, indent=2)

print("결과 저장 완료:", MODEL_DIR)

# **8. 직접 판별해 보기**


In [ ]:
# 글 여러 개를 한 번에 토큰화해 ChatGPT(1) 확률을 계산합니다.
@torch.no_grad()
def predict_scores(texts):
    encoded = best_tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LEN,
        padding=True,
        return_token_type_ids=False,
        return_tensors="pt",
    ).to(DEVICE)

    logits = best_model(**encoded).logits.float()
    return torch.softmax(logits, dim=1)[:, 1].cpu().tolist()


def split_sentences(text):
    # 마침표, 물음표, 느낌표 뒤 공백 또는 줄바꿈 기준으로 문장 분리
    pieces = re.split(r"(?<=[.!?])\s+|\n+", text.strip())
    return [piece.strip() for piece in pieces if piece.strip()]

In [ ]:
# 전체 글 의심 점수와 문장별(앞뒤 문맥 포함) 점수를 화면에 표시합니다.
def show_detection(text):
    text = clean_text(text)  # 학습 데이터와 같은 정리 규칙 적용
    if not text:
        raise ValueError("검사할 글을 입력")

    score = predict_scores([text])[0]
    token_count = len(best_tokenizer(text)["input_ids"])

    print(f"HC3-ko 기준 ChatGPT 의심 점수: {score * 100:.1f}%")

    if token_count > MAX_LEN:
        print(f"전체 {token_count}토큰 중 앞 {MAX_LEN}토큰만 전체 점수에 사용")

    sentences = split_sentences(text)
    if len(sentences) < 2:
        print("문장이 하나여서 문장별 표시는 생략")
        return

    # 각 문장과 앞뒤 문장을 묶어 한 번에 점수 계산
    contexts = [
        " ".join(sentences[max(0, i - CONTEXT_RADIUS): i + CONTEXT_RADIUS + 1])
        for i in range(len(sentences))
    ]
    local_scores = predict_scores(contexts)

    print("\n문장별 의심 점수:")
    blocks = []
    for sentence, local_score in zip(sentences, local_scores):
        color = "#ffe0e0" if local_score >= SUSPICION_THRESHOLD else "#e8f3e8"
        blocks.append(
            f'<div style="background:{color}; padding:8px; margin:5px 0; border-radius:5px;">'
            f'{html.escape(sentence)} <b>({local_score * 100:.1f}%)</b>'
            f'</div>'
        )

    display(HTML("".join(blocks)))

In [ ]:
# 직접 작성한 글을 넣어 판별 결과를 확인합니다.
my_text = """
여기에 직접 검사할 한국어 글을 입력하세요.
문장을 여러 개 입력하면 문장별 의심 점수도 함께 표시됩니다.
"""

show_detection(my_text)